In [ ]:
from dataloader import ValidationDatasetFromSet
from utils import ModelValidator
import torch
from torch.utils.data import DataLoader
from models import WavLM_Base_ECAPA_TDNN
import pandas as pd
import librosa
import random
from utils.distance import compute_distance, l2_normalize

def read_audio(filename, max_length=0):
    waveform, _ = librosa.load(filename, sr=16000)
    waveform, _ = librosa.effects.trim(waveform, top_db=35)
    waveform = torch.tensor(waveform, dtype=torch.float32)
    if max_length > 0 and waveform.shape[0] > max_length:
        start_sample = random.randint(0, waveform.shape[-1] - max_length)
        end_sample = start_sample + max_length
        waveform = waveform[start_sample:end_sample]
    return waveform

model = WavLM_Base_ECAPA_TDNN(frozen=False)
state = torch.load(
        f'../models/WavLM-Base-joint_ECAPA-TDNN_Random-Triplet-Mining_LibriSpeech-Genuine_best_model_state.pth')
model.load_state_dict(state)
model.to('cuda')
model.eval()

val_set = pd.read_csv("../validation_sets/BSI/valid.csv")

In [ ]:
# Function to extract embedding from the model
def extract_embedding(model, filename):
    waveform = read_audio(filename)  # Assuming read_audio function is already defined
    waveform = waveform.unsqueeze(0).to('cuda')  # Add batch dimension and move to GPU
    with torch.no_grad():
        embedding = model(waveform)
    return l2_normalize(embedding.cpu())

# Calculate distances and update val_set
distances = []
for index, row in val_set.iterrows():
    embedding_1 = extract_embedding(model, row['filename'])
    embedding_2 = extract_embedding(model, row['filename_to_check'])
    distance = compute_distance(embedding_1, embedding_2)
    distances.append(distance[0])

val_set['distance'] = distances

val_set.head()

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

# Calculate the false positive rate (FPR), true positive rate (TPR), and thresholds
fpr, tpr, thresholds = roc_curve(val_set['is_same_speaker'], val_set['distance'])

# Calculate the absolute difference between FPR and FNR
fnr = 1 - tpr
eer_index = np.nanargmin(np.absolute(fpr - fnr))

# Calculate EER and the corresponding threshold
eer = fpr[eer_index]
eer_threshold = thresholds[eer_index]

print(f"EER: {eer:.4f}")
print(f"Optimal threshold: {eer_threshold:.4f}")
